# 08. Penanganan Pencilan (Handling Outliers)

Pencilan adalah observasi yang menyimpang menurut kriteria statistik tertentu. Dalam data teknis, nilai ekstrem dapat merupakan kejadian nyata dan penting. Deteksi tidak sama dengan keputusan menghapus.

In [ ]:
import pandas as pd

def iqr_outlier_report(series: pd.Series) -> dict:
    values = pd.to_numeric(series, errors="coerce").dropna()
    if values.empty:
        return {"q1": None, "q3": None, "iqr": None,
                "lower_bound": None, "upper_bound": None, "count": 0}
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return {
        "q1": q1, "q3": q3, "iqr": iqr,
        "lower_bound": lower, "upper_bound": upper,
        "count": int(((values < lower) | (values > upper)).sum()),
    }

if "df_transformed" in globals():
    numeric_columns = df_transformed.select_dtypes(include="number").columns.tolist()
    outlier_rows = []
    for column in numeric_columns:
        report = iqr_outlier_report(df_transformed[column])
        report["column"] = column
        outlier_rows.append(report)
    display(pd.DataFrame(outlier_rows).set_index("column") if outlier_rows else pd.DataFrame())
else:
    print("Jalankan notebook 06 terlebih dahulu.")

## Keputusan perlakuan
Untuk setiap kandidat pencilan, periksa rekaman sumber dan dokumentasikan keputusan:
- Dipertahankan jika valid dan relevan.
- Dikoreksi hanya jika terbukti kesalahan ekstraksi/input dan nilai yang benar dapat ditelusuri.
- Dikecualikan hanya dengan alasan yang jelas, tercatat, dan disertai dampak terhadap sampel.
- Capping/winsorization atau transformasi tidak dilakukan secara otomatis.

Z-score dengan ambang seperti |Z| > 3 hanya digunakan dengan kehati-hatian dan asumsi distribusi yang sesuai. Jangan menghapus nilai ekstrem semata-mata agar grafik tampak rapi.